# 09 · Despliegue en AWS EC2

**Propósito.** Documentar cómo llevamos la aplicación de [08-contenedores](08-contenedores.ipynb) a una instancia EC2 y cómo comprobamos una predicción desde otra computadora, con la versión del modelo identificada, los logs y el cierre de recursos.

Este notebook no se conecta a AWS: registra el procedimiento, las decisiones y lo que observamos. No incluye IP, cuentas, correos ni llaves.

## 1. Quién hace qué

| Rol | Integrante |
|---|---|
| Administra la instancia EC2 (es su instancia asignada) | _(completar)_ |
| Ejecuta el despliegue remoto | _(completar)_ |
| Administra Databricks (experimento y Registry) | _(completar)_ |
| Prueba desde otra computadora | _(completar)_ |

Cada quien usa sus propios accesos. No compartimos la llave `.pem`, tokens ni `.env`. _(Si quien despliega no tiene acceso al workspace del modelo: completar aquí cómo se transfirió en privado el artefacto `artifacts/champion_model.pkl` + `champion_metadata.json`, su versión y run de origen, y que la API corrió con `MODEL_SOURCE=local`.)_

## 2. Instancia utilizada

Reutilizamos **una** instancia ya asignada; no creamos una nueva para esta entrega.

| Característica | Valor |
|---|---|
| Región | `us-east-1` (N. Virginia) |
| AMI | Ubuntu Server 24.04 LTS, x86_64 |
| Tipo | `t3.medium` (2 vCPU, 4 GiB) |
| Disco | 20 GiB gp3 |
| Carpeta del proyecto en la instancia | `~/proyecto/TubeCast` (separada de `~/pcd-entregas-2026` del laboratorio y de la del examen) |

Antes de construir revisamos capacidad y lo que ya existía, sin borrar nada de otras actividades:

```bash
df -h          # espacio en disco: las dos imágenes ocupan varios GB
free -h        # memoria
sudo docker ps -a
sudo docker images
```

**Resultado observado:** _(completar: espacio libre antes del build, contenedores previos encontrados y si estaban detenidos)_.

## 3. Grupo de seguridad (acceso limitado)

| Regla | Puerto | Origen | Para qué |
|---|---|---|---|
| SSH | 22 | *My IP* del responsable (`/32`) | Administrar la instancia |
| Custom TCP | 8501 | IP pública de las computadoras que prueban (`/32`) | Abrir Streamlit |
| — | 8000 | **sin regla** | La API escucha en `127.0.0.1`; sólo la alcanza la interfaz dentro del servidor |

Si cambia nuestra IP pública, actualizamos el origen de la regla SSH a *My IP*; nunca la abrimos a `0.0.0.0/0`. La IPv4 pública de la instancia cambia en cada arranque, así que la revisamos en la consola antes de conectarnos.

![Instancia y red](../evidencia/t8-03-ec2-instancia-red.png)

*Interpretación:* la instancia está `Running` en `us-east-1` y el grupo de seguridad sólo tiene las dos reglas de entrada necesarias, con orígenes `/32`. IP y cuenta ocultas.

## 4. Pasos reproducibles

*En EC2, terminal SSH* (`ssh -i "RUTA_LLAVE" ubuntu@IP_PUBLICA`). Docker y Compose ya estaban instalados desde la clase 20 (`sudo docker compose version`).

**4.1 Código.** El repositorio es privado. Usamos un *fine-grained token* temporal, de sólo lectura y limitado a `cesarsantos23/TubeCast`; Git lo pide como contraseña y no queda en la URL ni en el historial.

```bash
mkdir -p ~/proyecto && cd ~/proyecto
git -c credential.helper= clone https://github.com/cesarsantos23/TubeCast.git
cd TubeCast
git switch feat/tercera-entrega
git rev-parse --short HEAD
```

**4.2 Configuración privada**, creada en la instancia y fuera de Git:

```bash
cp .env.example .env
chmod 600 .env
nano .env        # accesos propios; API_HOST=127.0.0.1
```

**4.3 Construir y arrancar:**

```bash
sudo docker compose -f docker-compose.yaml config --quiet
sudo docker compose -f docker-compose.yaml up --build -d
sudo docker compose -f docker-compose.yaml ps
```

![Despliegue en EC2](../evidencia/t8-04-ec2-despliegue.png)

*Interpretación:* el commit desplegado es el de la rama `feat/tercera-entrega`; los dos servicios están arriba y la API `healthy`. **Resultado observado:** _(completar: commit corto y tiempo del build en EC2)_.

## 5. Versión servida y predicción remota

Desde el servidor, la versión que cargó la API:

```bash
curl http://127.0.0.1:8000/health
```

Desde **otra computadora** autorizada en el grupo de seguridad: `http://IP_PUBLICA:8501`, capturar los 14 valores y pulsar **Obtener Predicción**.

| Entrada (14 cohortes, `t-14 … t-1`) | Vistas estimadas | Versión | Familia | Línea en `logs api` |
|---|---|---|---|---|
| _(completar: valores usados)_ | _(completar)_ | _(completar: vN)_ | _(completar)_ | `Predicción: ... | versión=N` |

![Predicción remota](../evidencia/t8-05-prediccion-remota.png)

*Interpretación:* la predicción sale del modelo registrado: la versión que muestra la interfaz es la misma que reporta `/health` y la que aparece en el log de la petición, y coincide con la versión `champion` del Registry (notebook 04).

## 6. Entrada inválida en el servidor

La API no está expuesta a Internet, así que la validación se prueba por SSH:

```bash
curl -i -X POST http://127.0.0.1:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"lags_14_dias":[100,100,100,100,100,100,100,100,100,100,100,100,100,-5]}'
```

![Entrada inválida en EC2](../evidencia/t8-06-ec2-entrada-invalida.png)

*Interpretación:* la respuesta es `422 Unprocessable Content` con el mensaje de que las vistas no pueden ser negativas; no se devuelve ninguna predicción.

## 7. Logs

```bash
sudo docker compose -f docker-compose.yaml logs --tail 30 api
sudo docker compose -f docker-compose.yaml logs --tail 30 ui
```

![Logs](../evidencia/t8-07-logs.png)

*Interpretación:* en el arranque se ve `Modelo cargado: ... -> versión N` y `Application startup complete`; después, cada petición registra `Predicción: ... | versión=N` y su `POST /predict` con estado 200 (o 422 para la entrada inválida). Así relacionamos la predicción de la interfaz con la versión servida.

**Diagnóstico de un error:** _(completar si apareció alguno: qué decía el log, qué revisamos y qué cambiamos. Por ejemplo, si la API arranca en `sin_modelo`, `/health` muestra en `detail` si faltó un acceso o el nombre del modelo.)_

## 8. Cierre y costos

```bash
sudo docker compose -f docker-compose.yaml down
exit
```

En la consola de AWS: **Instance state → Stop instance** hasta ver `Stopped`. Después borramos el token temporal de GitHub.

![Cierre](../evidencia/t8-08-cierre.png)

| Recurso | Estado al terminar | ¿Genera costo? |
|---|---|---|
| Contenedores | retirados con `down` | No |
| Imágenes Docker y código en el disco | se conservan para volver a arrancar | Ocupan el EBS |
| Instancia `t3.medium` | `Stopped` (no terminada: es reutilizada) | No se cobra cómputo detenida |
| Volumen EBS 20 GiB gp3 | se conserva | **Sí**, por GB-mes |
| IPv4 pública | se libera al detener | No (sin IP elástica) |

No terminamos la instancia porque es la asignada a un integrante y se usará en la entrega final.

## 9. Después del merge

Una vez fusionado el PR, en la misma instancia:

```bash
cd ~/proyecto/TubeCast
git switch main
git pull                      # con el token temporal, si hace falta
git rev-parse --short HEAD
sudo docker compose -f docker-compose.yaml up --build -d
curl http://127.0.0.1:8000/health
```

**Resultado observado:** _(completar: commit de `main`, versión servida y que la predicción con la misma entrada da el mismo valor)_.

## 10. Limitaciones

- La interfaz se sirve por HTTP, sin certificado; aceptable para la demostración con orígenes restringidos.
- La API depende de Databricks al arrancar; si el workspace no responde, queda sin modelo (lo indica `/health`).
- La instancia no se queda encendida: para volver a probar hay que arrancarla, revisar la nueva IP pública y repetir `up -d`.